In [0]:
%pip install langchain langchain-community langchain-core mlflow --quiet
dbutils.library.restartPython()

In [0]:
from langchain_core.prompts import PromptTemplate
from mlflow.deployments import get_deploy_client

# Create a simple LLM wrapper for Databricks Foundation Models
class DatabricksLLM:
    def __init__(self, endpoint, max_tokens=500):
        self.endpoint = endpoint
        self.max_tokens = max_tokens
        self.client = get_deploy_client("databricks")
    
    def invoke(self, prompt_value):
        # Extract the text from the prompt
        if hasattr(prompt_value, 'text'):
            text = prompt_value.text
        else:
            text = str(prompt_value)
        
        # Call Databricks Foundation Model API
        response = self.client.predict(
            endpoint=self.endpoint,
            inputs={"messages": [{"role": "user", "content": text}]}
        )
        
        # Return response in a format compatible with LangChain
        class Response:
            def __init__(self, content):
                self.content = content
        
        return Response(response['choices'][0]['message']['content'])

# Initialize the Databricks Foundation Model
llm = DatabricksLLM(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=500)

# Create a prompt template for summarization
template = """
You are an expert at summarizing text. Please read the following text and provide a concise summary in 3-5 sentences.

Text:
{text}

Summary:
"""

prompt = PromptTemplate(
    input_variables=["text"],
    template=template
)

# Create a summarizer function using LangChain PromptTemplate
def summarize_text(text):
    # Format the prompt using the PromptTemplate
    formatted_prompt = prompt.format(text=text)
    
    # Call the LLM
    response = llm.client.predict(
        endpoint=llm.endpoint,
        inputs={"messages": [{"role": "user", "content": formatted_prompt}]}
    )
    
    return response['choices'][0]['message']['content']

# Example usage
sample_text = """
Artificial Intelligence (AI) has transformed numerous industries over the past decade. 
From healthcare to finance, AI technologies are being deployed to solve complex problems 
and improve efficiency. Machine learning algorithms can now analyze vast amounts of data 
to identify patterns and make predictions with remarkable accuracy. In healthcare, AI 
systems assist doctors in diagnosing diseases, while in finance, they help detect fraudulent 
transactions. Natural Language Processing (NLP) has enabled machines to understand and 
generate human language, leading to advanced chatbots and virtual assistants. Computer 
vision allows machines to interpret visual information, powering applications like facial 
recognition and autonomous vehicles. Despite these advances, challenges remain, including 
ethical concerns about bias in AI systems, privacy issues, and the need for explainable AI. 
As AI continues to evolve, researchers and policymakers are working to ensure that these 
technologies are developed and deployed responsibly, maximizing benefits while minimizing 
potential harms to society.
"""

# Generate summary using the function
import textwrap

result = summarize_text(sample_text)

# Display with better formatting
print("\n" + "="*80)
print("📄 ORIGINAL TEXT")
print("="*80)
print(f"Length: {len(sample_text)} characters\n")
wrapped_original = textwrap.fill(sample_text.strip(), width=75)
print(wrapped_original)

print("\n\n" + "="*80)
print("✨ SUMMARY")
print("="*80)
# Wrap the summary text for better readability
wrapped_summary = textwrap.fill(result, width=75)
print(f"\n{wrapped_summary}\n")
print("="*80)

In [0]:
# You can now use the summarizer with your own text
# Simply call the summarize_text function with any long text

# Example: Summarize your own text
your_text = """
LangChain Practice Assignments
Assignment 1: Email Summarizer
Objective: Summarize long emails into key points and action items.
Input: A long email (10–20 lines).
Expected Output: Summary (3 lines), Key action items, Priority (High/Medium/Low)
LangChain Concepts: PromptTemplate, LLMChain
Assignment 2: Meeting Minutes Generator
Objective: Generate meeting summary and extract action items from transcript.
Input: Meeting transcript text.
Expected Output: Meeting Summary, Decisions Taken, Action Items, Owners
LangChain Concepts: Prompt Engineering, Chains
Assignment 3: PDF Q&A; Bot
Objective: Answer questions based on PDF documents.
Input: PDF document + User question
Expected Output: Answer from the document with source page number
LangChain Concepts: Document Loader, Embeddings, Vector Store, RetrievalQA
Assignment 4: Chatbot with Memory
Objective: Build chatbot that remembers conversation context.
Input: Conversation input, e.g., "My name is Mahesh." Then later: "What is my name?"
Expected Output: "Your name is Mahesh."
LangChain Concepts: Memory, ConversationChain
Assignment 5: Product Description Generator
Objective: Generate product descriptions and marketing slogans.
Input: Product Name e.g., "Wireless Earbuds"
Expected Output: Product Description, Features, Marketing Tagline
LangChain Concepts: PromptTemplate, Chains
Assignment 6: FAQ Bot
Objective: Build FAQ retrieval system.
Input: FAQ document + User question e.g., "How do I reset my password?"
Expected Output: Answer from FAQ
LangChain Concepts: RAG, Retrieval
Assignment 7: SQL Agent
Objective: Execute natural language queries on a database.
Input: Database + Natural language query, e.g., "Show top 5 employees by salary."
Expected Output: Generated SQL and query results
LangChain Concepts: Agents, Tool Calling
Assignment 8: CSV Data Analyst
Objective: Analyze CSV data via natural language.
Input: CSV file containing sales data.
Expected Output: Answers: Total sales; Highest revenue month; Average order value
LangChain Concepts: DataFrame Agent, Tools
Assignment 9: Multi-Tool Agent
Objective: Use different tools (Calculator, Date Tool) via agent.
Input: Questions like "What is 200 * 45?", "What date is today?"
Expected Output: Calculator output and current date responses
LangChain Concepts: Agents, Tool Selection
Assignment 10: Resume Matcher
Objective: Match resumes against job descriptions.
Input: Job Description e.g., "Python, Azure, LangChain" and Candidate resume
Expected Output: Match Score (%), Matching Skills, Missing Skills, Recommendation
LangChain Concepts: Embeddings, Similarity Search
Assignment 11: Text Summarizer
Objective: Summarize a long text into concise summary.
Input: Long article or document text.
Expected Output: 3-5 sentence summary
LangChain Concepts: LLMChain, PromptTemplate
Assignment 12: Sentiment Analysis
Objective: Classify sentiment of user reviews.
Input: User review text.
Expected Output: Sentiment label: Positive, Neutral, or Negative
LangChain Concepts: PromptTemplate, LLMChain
Assignment 13: Translation Assistant
Objective: Translate text between languages.
Input: Text and target language
Expected Output: Translated text in target language
LangChain Concepts: LLMChain, Prompt Engineering
Assignment 14: Named Entity Extractor
Objective: Extract named entities from text.
Input: Any textual content (e.g., news article)
Expected Output: List of named entities (persons, organizations, locations)
LangChain Concepts: PromptTemplate, Chains
Assignment 15: Knowledge Base FAQ Retriever
Objective: Retrieve answers from a knowledge base.
Input: Knowledge base documents + User question
Expected Output: Answer from knowledge base with reference
LangChain Concepts: RAG, RetrievalQA
Assignment 16: Document Classification Agent
Objective: Classify documents into categories.
Input: Document text
Expected Output: Document category label (e.g., Finance, HR, Sales)
LangChain Concepts: LLMChain, Prompt Template
Assignment 17: Code Generation Assistant
Objective: Generate code snippets based on descriptions.
Input: Problem description
Expected Output: Code snippet solving the problem
LangChain Concepts: Chains, Prompt Engineering
Assignment 18: Resume Parser
Objective: Extract structured fields from resume text.
Input: Resume text
Expected Output: Parsed fields such as Name, Skills, Experience
LangChain Concepts: PromptTemplate, LLMChain
Assignment 19: Meeting Action Item Extractor
Objective: Extract action items and owners from meeting notes.
Input: Meeting notes text
Expected Output: List of action items with assigned owners
LangChain Concepts: Chains, PromptEngineering
Assignment 20: Content Rewriter
Objective: Rewrite content to match a specified tone.
Input: Original content and desired tone (e.g., formal, casual)
Expected Output: Rewritten content in desired tone
LangChain Concepts: PromptTemplate, LLMChain
"""

# Generate and display summary with formatting
import textwrap

result = summarize_text(your_text)

print("\n" + "="*80)
print("✨ SUMMARY")
print("="*80)
wrapped_summary = textwrap.fill(result, width=75)
print(f"\n{wrapped_summary}\n")
print("="*80)